# Hugging Face LLM Course Chapter 9.2  
## Building Your First Demo with Gradio

This notebook is adapted for classroom use from Hugging Face LLM Course Chapter 9.2: **Building your first demo**.

### Learning goals

By the end of this notebook, students should be able to:

1. Explain what Gradio is used for.
2. Wrap a normal Python function as a web demo.
3. Customize input components such as `gr.Textbox`.
4. Use a Hugging Face `transformers.pipeline()` inside a Gradio app.
5. Understand the basic deployment logic behind Hugging Face Spaces.

> Recommended environment: Google Colab  
> Suggested runtime: CPU is enough for the first examples; GPU is helpful but not required for small text generation.


## 0. Install dependencies

In Google Colab, run this cell first.

After installation, if Colab asks you to restart the runtime, restart it and run the notebook again from the top.


In [ ]:
!pip install -q gradio==4.44.1 transformers==4.41.2 torch

In [ ]:
import gradio as gr
import transformers

print("gradio:", gr.__version__)
print("transformers:", transformers.__version__)

## 1. What is Gradio?

Gradio is a Python library that helps us quickly build a web interface around a Python function.

The basic idea is:

```text
Python function
      ↓
Gradio Interface
      ↓
Interactive web demo
```

This is very useful for machine learning because we often already have a function such as:

```python
predict(input_text)
```

Gradio allows users to test that function through a simple web UI.


## 2. Hello World demo

The simplest Gradio demo has three key parts:

```python
gr.Interface(fn=..., inputs=..., outputs=...)
```

- `fn`: the Python function to run
- `inputs`: the input component
- `outputs`: the output component


In [ ]:
import gradio as gr

def greet(name):
    return "Hello " + name

demo = gr.Interface(
    fn=greet,
    inputs="text",
    outputs="text",
    title="Hello World Demo"
)

demo.launch()

### Try it

Type your name into the textbox and click **Submit**.

### Key idea

The function itself is very simple:

```python
def greet(name):
    return "Hello " + name
```

Gradio does not change the logic of the function.  
It only creates an interface so that other people can use the function easily.


## 3. Customize the input textbox

Instead of using the shortcut `"text"`, we can create a textbox component explicitly.

This allows us to set:

- `label`
- `placeholder`
- `lines`


In [ ]:
import gradio as gr

def greet(name):
    return "Hello " + name

textbox = gr.Textbox(
    label="Type your name here:",
    placeholder="John Doe",
    lines=2
)

demo = gr.Interface(
    fn=greet,
    inputs=textbox,
    outputs="text",
    title="Customized Textbox Demo"
)

demo.launch()

## 4. Exercise 1

Modify the function below so that it returns a more polite greeting.

For example:

```text
Good morning, Alice. Welcome to our AI demo!
```


In [ ]:
import gradio as gr

def polite_greet(name):
    # TODO: modify this function
    return "Hello " + name

demo = gr.Interface(
    fn=polite_greet,
    inputs=gr.Textbox(label="Name", placeholder="Alice"),
    outputs="text",
    title="Exercise 1: Polite Greeting"
)

demo.launch()

## 5. Including model predictions

Now we will connect a real NLP model to Gradio.

We will use Hugging Face `transformers.pipeline()`.

The general pattern is:

```python
from transformers import pipeline

model = pipeline("task-name")

def predict(input_text):
    result = model(input_text)
    return result
```

Then we wrap `predict()` with Gradio.


## 6. Load a small text-generation model

The course example uses a text-generation pipeline.  
For classroom use, we use `distilgpt2` because it is smaller than GPT-2 and usually faster to load.

The first time you run this cell, it will download the model.


In [ ]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="distilgpt2"
)

def predict(prompt):
    output = generator(
        prompt,
        max_new_tokens=40,
        num_return_sequences=1,
        do_sample=True,
        temperature=0.8
    )
    return output[0]["generated_text"]

In [ ]:
predict("My favorite programming language is")

## 7. Create a Gradio interface for text generation

Now we connect the `predict()` function to Gradio.


In [ ]:
import gradio as gr

demo = gr.Interface(
    fn=predict,
    inputs=gr.Textbox(
        label="Prompt",
        placeholder="My favorite programming language is",
        lines=3
    ),
    outputs=gr.Textbox(
        label="Generated Text",
        lines=6
    ),
    title="Text Generation Demo",
    description="This demo uses a small Transformer model through Hugging Face pipeline()."
)

demo.launch()

## 8. Important teaching explanation

This demo shows the deployment pattern, not model training.

In this notebook:

```text
We load an existing pretrained model.
We use the model to generate text.
We wrap the model prediction function with Gradio.
```

We are **not** fine-tuning the model here.

For your IMDb sentiment classifier deployment, the pattern will be very similar:

```text
Fine-tuned model on Hugging Face Hub
      ↓
pipeline("text-classification", model="your-username/your-model")
      ↓
predict(text)
      ↓
Gradio Interface
      ↓
Hugging Face Space
```


## 9. Optional: IMDb sentiment classifier demo

Use this section after you have uploaded your fine-tuned IMDb model to Hugging Face Hub.

Replace:

```text
YOUR_USERNAME/imdb-distilbert-demo
```

with your actual model repo.


In [ ]:
from transformers import pipeline

MODEL_ID = "YOUR_USERNAME/imdb-distilbert-demo"  # Change this

sentiment_classifier = pipeline(
    "text-classification",
    model=MODEL_ID,
    tokenizer=MODEL_ID
)

label_map = {
    "LABEL_0": "Negative",
    "LABEL_1": "Positive"
}

def classify_review(review):
    result = sentiment_classifier(review)[0]
    label = label_map.get(result["label"], result["label"])
    score = result["score"]
    return {
        "prediction": label,
        "confidence": round(score, 4)
    }

In [ ]:
# Example test
# This cell works only after MODEL_ID is changed to your real model repo.

# classify_review("This movie was wonderful. I really enjoyed it.")

In [ ]:
import gradio as gr

demo = gr.Interface(
    fn=classify_review,
    inputs=gr.Textbox(
        label="Movie Review",
        placeholder="Type a movie review here...",
        lines=5
    ),
    outputs=gr.JSON(label="Prediction"),
    title="IMDb Sentiment Classifier",
    description="This demo uses a fine-tuned DistilBERT model from Hugging Face Hub."
)

# Uncomment this after MODEL_ID is changed.
# demo.launch()

## 10. Exercise 2

Create your own Gradio demo for a simple Python function.

Examples:

1. A function that converts Celsius to Fahrenheit.
2. A function that counts the number of words in a sentence.
3. A function that checks whether a review is long or short.


In [ ]:
import gradio as gr

def word_count(text):
    words = text.split()
    return len(words)

demo = gr.Interface(
    fn=word_count,
    inputs=gr.Textbox(label="Input text", lines=4),
    outputs="number",
    title="Word Counter Demo"
)

demo.launch()

## 11. From notebook to Hugging Face Spaces

To deploy this as a Hugging Face Space, create a Space with SDK = Gradio, then add two files.

### `app.py`

```python
import gradio as gr
from transformers import pipeline

generator = pipeline("text-generation", model="distilgpt2")

def predict(prompt):
    output = generator(
        prompt,
        max_new_tokens=40,
        num_return_sequences=1,
        do_sample=True,
        temperature=0.8
    )
    return output[0]["generated_text"]

demo = gr.Interface(
    fn=predict,
    inputs=gr.Textbox(label="Prompt", lines=3),
    outputs=gr.Textbox(label="Generated Text", lines=6),
    title="Text Generation Demo"
)

if __name__ == "__main__":
    demo.launch()
```

### `requirements.txt`

```text
gradio==4.44.1
transformers==4.41.2
torch
```


## 12. Summary

In this notebook, we learned:

1. Gradio can wrap any Python function as a web demo.
2. `gr.Interface()` needs a function, inputs, and outputs.
3. Input components such as `gr.Textbox` can be customized.
4. Hugging Face `pipeline()` can be used inside a Gradio prediction function.
5. The same pattern can be used to deploy a fine-tuned IMDb sentiment model.

### Core pattern

```python
def predict(user_input):
    model_output = model(user_input)
    return model_output

demo = gr.Interface(fn=predict, inputs=..., outputs=...)
demo.launch()
```
